# Martingale control variates

For discounted payoff $X=e^{-rT}g(S_T)$ and a frozen adapted control, $I=\sum_j\widetilde Z_j^\top\Delta W_j$ has mean zero. Averaging $X-\beta I$ preserves the model price when the fitted model and coefficient are independent of evaluation draws.

## Analytical control

The five-asset geometric basket has an exact delta. Each grid uses the same terminal states and evaluates the Brownian integrand at the left endpoints. The integral mean and discrete Itô-isometry gap are in standard-error units.


In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
evidence = json.loads((root / "experiments/observations.json").read_text())
summary_path = root / "scratch/runs/final_suite/summary.json"
summary = json.loads(summary_path.read_text()) if summary_path.exists() else None

def show_table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

import sys

sys.path.insert(0, str(root))
from bsde_pricing.martingale import run_known_control_study
from bsde_pricing.problems import geometric_five_case

known = run_known_control_study(geometric_five_case(), n_paths=16_384, seed=2026,
                                steps=(4, 8, 16, 32, 64))
rows = []
for grid in known["grids"]:
    stat = grid["analytical_control"]
    rows.append((
        grid["steps"], f"{stat['estimate']:.6f}", f"{stat['standard_error']:.6f}",
        f"{grid['variance_reduction_factor']:.1f}×",
        f"{grid['stochastic_integral']['mean_in_standard_errors']:+.2f}",
        f"{grid['isometry_gap']['mean_in_standard_errors']:+.2f}",
    ))
show_table(["Intervals", "Price", "SE", "Variance reduction", "Integral mean / SE", "Isometry gap / SE"],
           rows)


| Intervals | Price | SE | Variance reduction | Integral mean / SE | Isometry gap / SE |
| --- | --- | --- | --- | --- | --- |
| 4 | 5.517769 | 0.019937 | 11.5× | -0.91 | -1.03 |
| 8 | 5.524709 | 0.014559 | 21.5× | -0.99 | -0.99 |
| 16 | 5.509509 | 0.010317 | 42.8× | -0.76 | -0.99 |
| 32 | 5.503657 | 0.007317 | 85.1× | -0.66 | -1.16 |
| 64 | 5.500169 | 0.005240 | 165.9× | -0.61 | -1.14 |


The exact price is 5.494719. Refining the grid from 4 to 64 intervals increases variance reduction from about 11× to 166×. The eight-interval estimate is 2.06 standard errors above the exact price. The integral means and isometry gaps are within 1.2 standard errors of zero. The grids share draws, so their errors are dependent.

## Matched precision for arithmetic ten

A method qualifies at the smallest measured allocation with the required half-width in all 16 repetitions for that training seed. The table charges fit, checkpoint setup and coefficient-pilot costs upfront, then charges evaluation for each fresh estimate. Its default rows cover the fully reconciled seed 41001. A local suite summary displays matched rows for each completed cohort.


In [ ]:
costs = evidence["estimators"]["cohort"]["matched"]
labels = {
    "antithetic_geometric_control": "Antithetic geometric payoff",
    "scrambled_sobol_geometric_control": "Sobol geometric payoff",
    "analytical_geometric_proxy_unit": "Geometric integral",
    "value_gradient_unit": "Value-gradient, unit",
    "value_gradient_fitted": "Value-gradient, fitted",
    "terminal_loss_unit": "Terminal loss, unit",
    "terminal_loss_fitted": "Terminal loss, fitted",
}
cohorts = {evidence["estimators"]["cohort"]["seed"]: costs}
if summary is not None:
    cohorts = {
        seed: {
            target: [(labels[method], row["evaluation_states"], row["total_seconds"]["1"],
                      row["total_seconds"]["100"])
                     for method, row in matches.items() if row["status"] == "target_met"]
            for target, matches in cohort["matched_precision"].items()
        }
        for seed, cohort in summary["estimators"].items() if "matched_precision" in cohort
    }

for seed, matches in cohorts.items():
    display(Markdown(f"**Training seed {seed}**"))
    show_table(
        ["Half-width", "Method", "Terminal states", "One estimate (s)", "100 estimates (s)"],
        [(target, name, f"{states:,}", f"{one:.4f}", f"{many:.4f}")
         for target, rows in matches.items() for name, states, one, many in rows],
    )


**Training seed 41001**

| Half-width | Method | Terminal states | One estimate (s) | 100 estimates (s) |
| --- | --- | --- | --- | --- |
| 0.025 | Antithetic geometric payoff | 8,192 | 0.0081 | 0.3565 |
| 0.025 | Sobol geometric payoff | 8,192 | 0.0707 | 6.6312 |
| 0.025 | Geometric integral | 32,768 | 1.8410 | 184.0976 |
| 0.025 | Value-gradient, unit | 32,768 | 35.4568 | 410.1013 |
| 0.025 | Value-gradient, fitted | 32,768 | 36.3779 | 411.0225 |
| 0.025 | Terminal loss, unit | 32,768 | 50.0926 | 231.3047 |
| 0.025 | Terminal loss, fitted | 32,768 | 50.6124 | 231.8244 |
| 0.01 | Antithetic geometric payoff | 32,768 | 0.0171 | 1.2640 |
| 0.01 | Sobol geometric payoff | 32,768 | 0.0693 | 6.4859 |
| 0.01 | Geometric integral | 131,072 | 7.7505 | 775.0544 |


At the 0.025 half-width target, the learned controls required 32,768 terminal states. Both conventional payoff controls qualified at 8,192 states and had lower marginal costs. Reusing the learned fit therefore gave no sustained break-even in this cohort. At 0.01, both conventional methods qualified at 32,768 states, the geometric integral at 131,072, and every learned variant missed the target.

These timings describe one training seed and one fixed contract. The execution workspace became unavailable after 31 of the 48 scheduled repetitions had been verified. [Recorded observations](../experiments/observations.json)
